In [1]:
import pandas as pd
from sqlalchemy import create_engine, inspect, URL

server = 'LAPTOP-AAS7RBMP'
data_warehouse_db = 'EcomerceDB'
open_data_db = 'EcomerceOpenDataDB'

warehouse_connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=data_warehouse_db,
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "trusted_connection": "yes"
    }
)

open_data_connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=open_data_db,
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "trusted_connection": "yes"
    }
)

warehouse_engine = create_engine(warehouse_connection_url)
open_data_engine = create_engine(open_data_connection_url)


In [9]:
# Extract data from data warehouse
print("Extracting data from Data Warehouse...")
sales_fact_df = pd.read_sql("SELECT * FROM SalesFact", warehouse_engine)
inventory_fact_df = pd.read_sql("SELECT * FROM InventoryFact", warehouse_engine)
customer_dim_df = pd.read_sql("SELECT * FROM CustomerDimension", warehouse_engine)
product_dim_df = pd.read_sql("SELECT * FROM ProductDimension", warehouse_engine)
date_dim_df = pd.read_sql("SELECT * FROM DateDimension", warehouse_engine)
shipper_dim_df = pd.read_sql("SELECT * FROM ShipperDimension", warehouse_engine)
supplier_dim_df = pd.read_sql("SELECT * FROM SupplierDimension", warehouse_engine)
print("Data extraction completed.")


Extracting data from Data Warehouse...
Data extraction completed.


In [13]:
# Merge SalesFact with relevant dimensions
sales_merged = sales_fact_df.merge(product_dim_df, on='ProductID', how='left') \
                           .merge(customer_dim_df, on='CustomerID', how='left') \
                           .merge(date_dim_df, on='DateID', how='left') \
                           .merge(shipper_dim_df, on='ShipperID', how='left') \
                           .merge(supplier_dim_df, on='SupplierID', how='left')

# Merge InventoryFact with relevant dimensions
inventory_merged = inventory_fact_df.merge(product_dim_df, on='ProductID', how='left') \
                                   .merge(supplier_dim_df, on='SupplierID', how='left') \
                                   .merge(date_dim_df, on='DateID', how='left') 


In [30]:
# Define PII columns to remove
pii_columns_sales = ['CustomerName', 'CustomerEmail','SupplierName','ShipperName','SupplierContact']
pii_columns_inventory = ['SupplierName', 'SupplierContact']

# Remove PII from Sales Fact
for col in pii_columns_sales:
    if col in sales_merged.columns:
        sales_merged.drop([col], axis=1, inplace=True)

# Remove PII from Inventory Fact
for col in pii_columns_inventory:
    if col in inventory_merged.columns:
        inventory_merged.drop([col], axis=1, inplace=True)


In [39]:
sales_merged['Year'] = sales_merged['Year'].astype('Int64')  # Use Int64 to allow for potential NaNs
sales_merged['Month'] = sales_merged['Month'].astype('Int64')
sales_merged['Day'] = sales_merged['Day'].astype('Int64')
sales_merged['Quarter'] = sales_merged['Quarter'].astype('Int64')

# Convert FullDate from object to datetime64
sales_merged['FullDate'] = pd.to_datetime(sales_merged['FullDate'], errors='coerce')
sales_merged['TotalAmount'] = sales_merged['TotalAmount'].round(2)
sales_merged['DiscountAmount'] = sales_merged['DiscountAmount'].round(2)
sales_merged['NetAmount'] = sales_merged['NetAmount'].round(2)
sales_merged['ProductPrice'] = sales_merged['ProductPrice'].round(2)

In [44]:
import pandas as pd

# Convert Year, Month, Day, and Quarter from float64 to int64
inventory_merged['Year'] = inventory_merged['Year'].astype('Int64')  # Use Int64 to handle any potential NaNs
inventory_merged['Month'] = inventory_merged['Month'].astype('Int64')
inventory_merged['Day'] = inventory_merged['Day'].astype('Int64')
inventory_merged['Quarter'] = inventory_merged['Quarter'].astype('Int64')

# Convert FullDate from object to datetime64
inventory_merged['FullDate'] = pd.to_datetime(inventory_merged['FullDate'], errors='coerce')

# Format ProductPrice as Decimal with 2 decimal places
inventory_merged['ProductPrice'] = inventory_merged['ProductPrice'].round(2)


In [42]:
sales_merged = sales_merged.drop(columns=['SalesID'])

# Now insert the data into the Sales table
sales_merged.to_sql('Sales', con=open_data_engine, if_exists='append', index=False)

60

In [45]:
inventory_merged.to_sql('Inventory', con=open_data_engine, if_exists='append', index=False)


90